In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
from tqdm import tqdm
from utils.my_config import *

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F
from utils.my_tokenizer import Tokenizer

print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"PyTorch sees {torch.cuda.device_count()} GPU(s)")
print(f"Current device index: {torch.cuda.current_device()}")
print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# # convert data from .dat to .npy

# import pandas as pd
# from tqdm import tqdm
# import numpy as np
# import utils.my_ecg_process as ecg

# info = pd.read_csv('data/ptbxl_database.csv', index_col='ecg_id')
# full_data = ecg.load_raw_data(info, "data/", sampling_rate, target_rate)
# np.save(full_data_path, full_data)

In [ ]:
import utils.my_ecg_process as ecg

if not os.path.exists(processed_data_path):
    dataset = np.load(full_data_path)
    processed_data = ecg.process_loaded_data(dataset, seq_length, total_length, target_rate)
    np.save(processed_data_path, processed_data)

else:
    processed_data = np.load(processed_data_path)

In [ ]:
def split_data(data, train_ratio=0.8, random_seed=42):
    n = data.shape[0]
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    split_idx = int(n * train_ratio)
    train_indices = indices[:split_idx]
    valid_indices = indices[split_idx:]
    
    train_data = data[train_indices]
    valid_data = data[valid_indices]
    
    return train_data, valid_data

In [ ]:
batch_size = 32

train, valid = split_data(processed_data, train_ratio=0.8, random_seed=42)
train_dataset = TensorDataset(torch.tensor(train, dtype=torch.float32))
valid_dataset = TensorDataset(torch.tensor(valid, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

In [ ]:
beat = Tokenizer(**signal_cfg).to(device)
optimizer = torch.optim.AdamW(beat.parameters(), lr=1e-4, weight_decay=0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

In [ ]:
writer = SummaryWriter(log_dir=f"runs/{beat_dir}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

train_idx = 5
valid_idx = 6
lead_names = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

def plot_one_figure(x, recon, pred, epoch, plot_type):
    x = x.cpu().detach().numpy()[0]
    recon = recon.cpu().detach().numpy()[0]
    pred = pred.cpu().detach().numpy()[0]
    ecg_data = np.hstack([x, recon, pred])
    mpl.rcParams.update({
        "font.family": "serif",    
        "font.serif": ["Helvetica", "DejaVu Serif"], 
        "font.weight": "medium",
        "font.size": 12,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "legend.fontsize": 13,
        "lines.linewidth": 1.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.formatter.use_mathtext": True,
    })

    colors = {
        "original": "#FF8C00",
        "reconstructed": "#00D0A8",
        "predicted": "#9C6ADE",
        "divider": "#2A5DFF"
    }

    line_styles = {
        "original": "-",
        "reconstructed": "-",
        "predicted": "-"
    }

    def plot_channel(ax, x, recon, pred, channel_idx):
        x = x[channel_idx]
        recon = recon[channel_idx]
        pred = pred[channel_idx]

        total_length = len(recon) + len(pred)
        time_axis = np.arange(total_length)

        ax.plot(time_axis[:len(x)], x[:total_length], 
                label="Original", 
                color=colors["original"],
                linestyle=line_styles["original"],
                alpha=0.8)
        
        ax.plot(time_axis[:len(recon)], recon, 
                label="Reconstructed", 
                color=colors["reconstructed"],
                linestyle=line_styles["reconstructed"],
                alpha=0.8)
        
        ax.plot(time_axis[len(recon):], pred, 
                label="Predicted", 
                color=colors["predicted"],
                linestyle=line_styles["predicted"],
                alpha=0.8)
        
        ax.axvline(x=seq_length, color=colors["divider"], linestyle='--', linewidth=2.5, alpha=0.9)

        ax.text(0.02, 0.9, lead_names[channel_idx], fontsize=16, 
                weight='medium',
                transform=ax.transAxes, bbox=dict(edgecolor='none', alpha=0))

        if channel_idx == 0:
            ax.legend(loc='lower left', frameon=False, fontsize='small')

        ax.set_ylim(np.min(ecg_data), np.max(ecg_data))
        xticks = np.arange(0, total_length+1, target_rate)
        ax.set_xticks(xticks)
        ax.set_yticks(np.linspace(np.min(ecg_data), np.max(ecg_data), 8))
        ax.set_yticklabels([])

        if channel_idx % 6 == 5:
            # ax.set_xticklabels([f"{x/250:.2f}" for x in xticks])
            labels = [f"{x/250:.2f}" if i%2==0 else "" for i, x in enumerate(xticks)]
            ax.set_xticklabels(labels)
            ax.set_xlabel('Time (s)', fontweight="medium")
        else:
            ax.set_xticklabels([])

        ax.grid(True, linestyle='-', linewidth=0.8, alpha=0.8)
        ax.set_facecolor('#fdfdfd')

    fig = plt.gcf()
    
    if plot_type == 1:
        fig.text(0.25, 0.96, f"Train Sample {train_idx} (Epoch {epoch})", 
                ha='center', va='center', fontsize=20)
    else:
        fig.text(0.75, 0.96, f"Valid Sample {valid_idx} (Epoch {epoch})", 
                ha='center', va='center', fontsize=20)

    for i in range(12):
        row = int(i % 6)
        col = int(i / 6)
        ax = plt.subplot(6, 4, row * 4 + col + plot_type * 2 - 1)
        plot_channel(ax, x, recon, pred, i)

def plot_signal(tokenizer, epoch):
    x_train = train_dataset[train_idx][0].unsqueeze(0).to(device)
    x_valid = valid_dataset[valid_idx][0].unsqueeze(0).to(device)
    with torch.no_grad():
        _, _, recon_train, pred_train, _ = tokenizer(x_train[:, :, :seq_length])
        _, _, recon_valid, pred_valid, _ = tokenizer(x_valid[:, :, :seq_length])

    plt.figure(figsize=(24, 18))
    plot_one_figure(x_train, recon_train, pred_train, epoch, 1)
    plot_one_figure(x_valid, recon_valid, pred_valid, epoch, 2)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    writer.add_figure(f"ECG/epoch_{epoch+1}", plt.gcf(), epoch)
    plt.close()

In [ ]:
num_epochs = 50
start_epoch = 0

os.makedirs(beat_ckpt_dir, exist_ok=True)

ckpt_files = [f for f in os.listdir(beat_ckpt_dir) if f.endswith(".pth")]
start_epoch = 0

if ckpt_files:
    ckpt_files.sort(key=lambda x: int(x.split("_")[1].split(".")[0]))
    latest_ckpt = os.path.join(beat_ckpt_dir, ckpt_files[-1])
    checkpoint = torch.load(latest_ckpt, map_location=device)
    
    beat.load_state_dict(checkpoint['model_state_dict'], strict=False)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Resuming from Epoch {start_epoch+1}")
else:
    print("No checkpoint found, start training from scratch")

In [ ]:
def compute_codebook_utility(indices, residual_levels, codebook_size):
    codebook_utility = np.zeros((residual_levels, codebook_size))
    for level in range(residual_levels):
        for i in range(indices.shape[0]):
            for j in range(indices.shape[1]):
                codebook_utility[level, indices[i, j, level]] = 1
    avg_codebook_utility = []
    for i in range(residual_levels):
        avg_codebook_utility.append(np.sum(codebook_utility[i, :])/codebook_size*100)
    return avg_codebook_utility

In [ ]:
λ_recon = 1.0
λ_pred = 0.5
λ_vq = 0.25

In [ ]:
for epoch in range(start_epoch, num_epochs):
    beat.train()
    with tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", unit="batch", dynamic_ncols=True) as tepoch:
        # recon_loss = 0
        # pred_loss = 0
        train_loss = 0
        for batch in tepoch:
            x = batch[0].to(device)
            optimizer.zero_grad()
            r_loss, vq_loss, _, pred_sequence, indices = beat(x[:, :, :seq_length])
            p_loss = F.mse_loss(pred_sequence, x[:, :, seq_length:seq_length+pred_sequence.shape[2]])
            loss = λ_recon * r_loss + λ_pred * p_loss + λ_vq * vq_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(beat.parameters(), max_norm=1.0)
            optimizer.step()
            # recon_loss += r_loss.detach().item()
            # pred_loss += p_loss.detach().item()
            train_loss += loss.detach().item()
            indices = indices.cpu().detach().numpy()
            tepoch.set_postfix(total_loss=loss.item())

    # avg_train_recon_loss = recon_loss / len(train_loader)
    # avg_train_pred_loss = pred_loss / len(train_loader)
    avg_train_loss = train_loss / len(train_loader)
    avg_train_codebook_utility = compute_codebook_utility(indices, residual_levels, codebook_size)
    scheduler.step()

    beat.eval()
    # recon_loss = 0
    # pred_loss = 0
    valid_loss = 0
    with torch.no_grad():
        for batch in valid_loader:
            x = batch[0].to(device)
            r_loss, vq_avg_loss, _, pred_sequence, indices = beat(x[:, :, :seq_length])
            p_loss = F.mse_loss(pred_sequence, x[:, :, seq_length:seq_length+pred_sequence.shape[2]])
            loss = λ_recon * r_loss + λ_pred * p_loss + λ_vq * vq_loss
            # recon_loss += r_loss.detach().item()
            # pred_loss += p_loss.detach().item()
            valid_loss += loss.detach().item()
            indices = indices.cpu().detach().numpy()
        
    # avg_valid_recon_loss = recon_loss / len(valid_loader)
    # avg_valid_pred_loss = pred_loss / len(valid_loader)
    avg_valid_loss = valid_loss / len(valid_loader)
    avg_valid_codebook_utility = compute_codebook_utility(indices, residual_levels, codebook_size)
    
    writer.add_scalars('Loss/Total', {'Train': avg_train_loss, 'Valid': avg_valid_loss}, epoch+1)
    # writer.add_scalars('Loss/Recon', {'Train': avg_train_recon_loss, 'Valid': avg_valid_recon_loss}, epoch+1)
    # writer.add_scalars('Loss/Pred', {'Train': avg_train_pred_loss, 'Valid': avg_valid_pred_loss}, epoch+1)
    for i in range(residual_levels):
        writer.add_scalars(f'Codebook_Utility/Level_{i+1}', {'Train': avg_train_codebook_utility[i], 'Valid': avg_valid_codebook_utility[i]}, epoch+1)
    if (epoch+1) % 5 == 0:
        plot_signal(beat, epoch)
        model = {
            'model_state_dict': beat.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'epoch': epoch+1
        }
        torch.save(model, f"{beat_ckpt_dir}/epoch_{epoch+1}.pth")
    
model = {
    'model_state_dict': beat.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'epoch':num_epochs+1
}
torch.save(model, tokenizer_path)